⚠️ **Under construction.** <!-- banner:under-construction --> This lab is drafted but not yet instructor-reviewed; it may change without notice until its session.

# Lab 14 · Multiplying many small numbers (and how not to)

**Today:** when you walk out you know what a product of many probabilities does to a computer, and the log trick that survives it — plus `and`/`or` for combining questions.

**Before you start:** chapter 5 read; this is its log-space section as pure mechanics.

Each section names one idea, explains what it does, and asks you to **predict
what a cell prints before you run it**. Write the prediction down — on paper,
out loud, in a comment. A wrong prediction you wrote down teaches you exactly
one thing; a shrug teaches you nothing.

Most sections end with a **Test your understanding** task: write a small piece
of code, then run the check cell under it. The check never grades and never
breaks anything — ⬜ means not attempted yet, ❌ means not yet (with a hint),
✅ means passing. The check cells are the one thing here to run rather than
edit; everything else is yours to break.

**AI in this lab:** until your prediction is written down, work at level 1 — no
AI. The prediction is how you find out what you, unaided, can already read, and
both exams are level 1. Once you have run a cell, level 3 is encouraged: ask
your tutor to explain any miss.

Run every cell. Change things. Breaking this notebook costs nothing and teaches
more than reading it.

## 1 · and / or

Two comparisons combine with plain words: `and` needs both true, `or` either. (These are for single True/False values — the mask versions from lab 10, `&` and `|`, are their elementwise cousins.)

**Predict which lines print.**

In [ ]:
count = 47
quality = 0.92
if count > 30 and quality > 0.9:
    print("usable")
if count > 100 or quality > 0.9:
    print("at least one credential")
if count > 100 and quality > 0.9:
    print("both credentials")

The first two print; the third does not. One habit worth copying: Python stops early — in an `and`, if the first half is already False, the second is never even evaluated.

**Test your understanding.** Write `is_reportable(count, quality, flagged)`: reportable means count at least 30 **and** quality above 0.9 **and not** flagged (use `flagged == False`).

In [ ]:
# your turn: is_reportable(count, quality, flagged)

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("is_reportable", expect=True, args=(47, 0.92, False))
check("is_reportable", expect=False, args=(47, 0.92, True))
check("is_reportable", expect=False, args=(29, 0.99, False),
      hint="at least 30 — is 29 in?")

## 2 · The product that vanishes

Multiply many probabilities — each below 1 — and the product plunges toward zero. The computer has a floor: below about 1e-308, it silently gives up and stores exactly 0.0. **Predict: which of the three products survives?**

In [ ]:
for n_factors in (100, 500, 2000):
    product = 1.0
    for _ in range(n_factors):
        product = product * 0.3
    print(n_factors, "factors:", product)

100 factors: about 5e-53 — tiny but alive. 500: about 4e-262 — barely. 2000: exactly **0.0** — not small, gone, and no error announced it. Any code that multiplies enough probabilities *will* cross the floor; the question is only when.

**Test your understanding.** Write `smallest_survivor(factor)`: multiply `factor` into a running product until the product becomes exactly 0.0, and return how many multiplications that took. (A `for` over `range(1, 100001)` with a `return` inside — lab 3's early-exit move.)

In [ ]:
# your turn: smallest_survivor(factor) — multiplications until exact 0.0

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("smallest_survivor", expect=619, args=(0.3,),
      hint="product = product * 0.3 each pass; return the pass number the moment product == 0.0")
check("smallest_survivor", expect=380, args=(0.14,))

## 3 · Logs turn the product into a sum

`np.log` gives a number's logarithm, and one property does all the work: **log(a·b) = log(a) + log(b)**. A product of 2,000 factors becomes a sum of 2,000 moderate negatives — no floor anywhere near. And logs keep order: bigger stays bigger. **Predict: what is the log-scale version of section 2's dead product, roughly?** (2,000 copies of log(0.3) ≈ −1.2 each.)

In [ ]:
import numpy as np

log_total = 0.0
for _ in range(2000):
    log_total += np.log(0.3)
print(round(log_total, 1))

About −2408 — a perfectly healthy number standing in for a value the product form could not hold at all. Chapter 5 classifies at 2,000 features on exactly this trick: compare log-sums, never products.

**Test your understanding.** Two log-scores: `score_a = -1206.8`, `score_b = -1612.5` (chapter 5's own). Set `winner` to the string `"a"` or `"b"` — decided *in code* with a comparison, not by eye — and `log_gap` to how much the winner beats the loser by (a positive number, 1 decimal).

In [ ]:
# your turn: winner and log_gap from score_a and score_b
score_a = -1206.8
score_b = -1612.5

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("winner", expect="a",
      hint="less negative is larger — the comparison is score_a > score_b")
check("log_gap", expect=405.7)

## If you finish early

- Chapter 5's `log_likelihood` is section 3 wearing per-feature factors — read it line against line.
- What does `smallest_survivor(0.99)` return, and why so much larger? Predict before running.

## If you are stuck

Wave someone over — this hour exists so a stuck step costs you a minute rather
than an evening. Known snags:

- **`smallest_survivor` returns None** — the check calls it with factors that do reach 0.0 within 100,000 passes; make sure the `if` is *inside* the loop and compares to exactly `0.0`.
- **`log_gap` is negative** — the task wants the winner's margin: subtract the smaller from the larger.
- **`and` on masks errored** — that is lab 10's territory: single values take `and`, arrays take `&`.